# Heatwave Detection and Analysis — Ouagadougou (1991–2024)

**Data:** ERA5-Land daily maximum 2-metre temperature (`t2m`)  
**Reference period:** 1991–2020 (WMO standard climatological normal)  
**Analysis year:** 2024

---

## Overview

This notebook detects and characterises heatwave events over Ouagadougou using ERA5-Land reanalysis data. Grid cells covering the city are averaged using cosine-latitude weighting to produce a single daily time series. The 90th-percentile threshold is computed for each calendar day from the 1991–2020 baseline and smoothed with a 30-day centred rolling mean to remove noise. A heatwave is then defined as a period of at least 3 consecutive days during which daily maximum 2-metre temperature exceeds this threshold (Perkins & Alexander, 2013). Intensity is expressed as degrees Celsius above the local threshold.

The workflow is organised in six sections:

1. **Setup** — imports and figure style  
2. **Data loading & exploration** — inspect the ERA5-Land dataset  
3. **Pre-processing** — reduce dimensions, unit conversion, build the time series  
4. **Threshold computation** — 90th-percentile climatology  
5. **Heatwave detection** — event identification and characterisation  
6. **Results & visualisation** — timeline, annual statistics, and trend analysis

> **Input file required:** `Ouagadougou_2001_2024_daily_tmax.nc`  
> Generated by the companion download notebook `ERA5_Daily_Temperature_Ouagadougou.ipynb`.

---
## 1 · Setup

In [ ]:
# Install optional dependencies (comment out after first run)
%pip install -q scipy

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.ndimage import label
from scipy.stats import linregress

# Reproducible figure style
%config InlineBackend.figure_format = 'retina'
plt.rcParams.update({
    'figure.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Constants
FNAME        = 'Ouagadougou_2001_2024_daily_tmax.nc'
REF_START    = '1991'
REF_END      = '2020'
TARGET_YEAR  = '2024'
KELVIN       = 273.15
PCT          = 90
MIN_DURATION = 3
WINDOW       = 30

print('All libraries imported.')

---
## 2 · Data Loading & Exploration

In [ ]:
ds_raw = xr.open_dataset(FNAME)
print(ds_raw)
print('\nTime range:',
      str(ds_raw.valid_time.min().values)[:10], '-',
      str(ds_raw.valid_time.max().values)[:10])
print('NaN count in t2m:', int(ds_raw.t2m.isnull().sum()))

In [ ]:
# Spatial snapshot — first day
# ERA5-Land daily-statistics files contain a spurious auxiliary 'time'
# dimension (an encoding artefact). Index 0 selects a single slice.
fig, ax = plt.subplots(figsize=(5, 4))
ds_raw.t2m.isel(valid_time=0, time=0).plot(ax=ax, cmap='plasma')
ax.set_title('ERA5-Land 2m Tmax — ' + str(ds_raw.valid_time.values[0])[:10])
plt.tight_layout()
plt.show()

---
## 3 · Pre-Processing

The raw dataset has an auxiliary `time` dimension encoding intra-day sub-steps (an artefact of the daily-statistics derivation). We average over it to obtain one value per calendar day (`valid_time`), then build a cosine-latitude-weighted spatial-mean time series in °C.

In [ ]:
# Collapse auxiliary 'time' dimension
ds = ds_raw.mean(dim='time', skipna=True)

# Cosine-latitude area-weighted spatial mean -> 1-D time series in Celsius
weights = np.cos(np.deg2rad(ds.latitude))
data_ts = (
    ds.weighted(weights)
      .mean(dim=['longitude', 'latitude'])
    - KELVIN
).compute()

# Attach day-of-year coordinate for groupby operations
data_ts = data_ts.assign_coords(
    dayofyear=('valid_time', data_ts.valid_time.dt.dayofyear.values)
)

print('Time series length:', data_ts.t2m.shape[0])
print(f'Tmax range: {float(data_ts.t2m.min()):.1f} C  to  {float(data_ts.t2m.max()):.1f} C')

In [ ]:
# Monthly climatology maps (1991-2020)
ds_clim      = ds.sel(valid_time=slice(REF_START, REF_END))
monthly_clim = ds_clim.t2m.groupby('valid_time.month').mean() - KELVIN

MONTH_LABELS = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']
vmin, vmax   = float(monthly_clim.min()), float(monthly_clim.max())

fig, axes = plt.subplots(
    3, 4, figsize=(16, 9),
    subplot_kw={'projection': ccrs.PlateCarree()},
    sharey=True,
)
for i, ax in enumerate(axes.flat):
    monthly_clim[i].plot(
        ax=ax, transform=ccrs.PlateCarree(),
        cmap='plasma', vmin=vmin, vmax=vmax,
        cbar_kwargs={'label': 'C'},
    )
    ax.set_title(MONTH_LABELS[i], fontsize=11)
    ax.coastlines(linewidth=0.7)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)

plt.suptitle(
    f'Monthly Climatology of Daily Maximum 2m Temperature ({REF_START}-{REF_END})',
    fontsize=14, fontweight='bold',
)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('monthly_climatology_tmax.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4 · Threshold Computation

For each calendar day (1–366) we compute the **90th percentile** of all values within a ±15-day window centred on that day, pooled across all reference years (1991–2020). The resulting 366-element array is then smoothed with a 30-day centred rolling mean to remove edge noise.

This approach follows Perkins & Alexander (2013) and avoids the step-function artefact produced by a simple `groupby.quantile` without smoothing.

In [ ]:
def compute_rolling_percentile(da, half_window=15, percentile=90.0, time_dim='valid_time'):
    """
    Compute a calendar-day percentile threshold using a centred moving window.

    For each day of year (1-366), values from all years within
    +/- half_window days of that calendar day are pooled and the
    percentile-th quantile is returned. Wrap-around is applied
    at the Dec-Jan boundary.

    Parameters
    ----------
    da          : Daily DataArray with a datetime-like time_dim coordinate.
    half_window : Half-width of the moving window in days. Default 15.
    percentile  : Percentile to compute (0-100). Default 90.
    time_dim    : Name of the time dimension. Default 'valid_time'.

    Returns
    -------
    xr.DataArray with dimension 'dayofyear' (1-366).
    """
    da = da.copy()
    da.coords['dayofyear'] = da[time_dim].dt.dayofyear
    results = []
    for doy in range(1, 367):
        window_days = np.arange(doy - half_window, doy + half_window + 1)
        window_days = (window_days - 1) % 366 + 1
        subset = da.sel({time_dim: da['dayofyear'].isin(window_days)})
        q = subset.reduce(np.nanpercentile, q=percentile, dim=time_dim)
        results.append(q)
    out = xr.concat(results, dim='dayofyear')
    return out.assign_coords(dayofyear=np.arange(1, 367))

In [ ]:
# Baseline time series
normal_ts = data_ts.sel(valid_time=slice(REF_START, REF_END))

# Method A — groupby quantile + rolling smooth  (used for detection)
normal_90     = normal_ts.groupby('valid_time.dayofyear').quantile(PCT / 100)
threshold_doy = normal_90.rolling(dayofyear=WINDOW, center=True, min_periods=1).mean()

# Method B — explicit moving-window percentile (for comparison)
threshold_mw = compute_rolling_percentile(normal_ts.t2m, half_window=15)

print(f'{PCT}th-percentile thresholds computed for all 366 calendar days.')

In [ ]:
# Compare threshold methods
fig, ax = plt.subplots(figsize=(11, 4))
threshold_doy.t2m.plot(
    ax=ax, color='crimson', lw=2,
    label=f'{PCT}th pct - groupby + {WINDOW}-day rolling (used for detection)'
)
threshold_mw.t2m.plot(
    ax=ax, color='steelblue', lw=1.8, ls='--',
    label=f'{PCT}th pct - explicit moving window +/-15 days'
)
ax.set_title(f'{PCT}th-percentile Threshold Comparison ({REF_START}-{REF_END})',
             fontweight='bold')
ax.set_xlabel('Day of Year')
ax.set_ylabel('Temperature (C)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('threshold_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Method A is used for all subsequent analyses
THRESHOLD = threshold_doy

---
## 5 · Heatwave Detection

### 5.1 · Detection Function

In [ ]:
def detect_heatwaves(exceedance_mask, min_duration=3):
    """
    Retain only threshold-exceedance runs lasting at least min_duration days.

    Parameters
    ----------
    exceedance_mask : Boolean array, True where temperature exceeds threshold.
    min_duration    : Minimum consecutive days to qualify as a heatwave.

    Returns
    -------
    Boolean array of the same shape; True only for qualifying heatwave days.
    """
    arr           = exceedance_mask.astype(int)
    group_labels, _ = label(arr)
    counts        = np.bincount(group_labels)
    valid         = np.isin(group_labels, np.where(counts >= min_duration)[0])
    return valid & exceedance_mask

### 5.2 · Visualise Heatwaves for the Target Year

In [ ]:
# Extract target-year values and align threshold
t2m_2024    = data_ts.t2m.sel(valid_time=TARGET_YEAR)
doy_2024    = t2m_2024.valid_time.dt.dayofyear.values
t2m_vals    = t2m_2024.values
thresh_vals = THRESHOLD.t2m.values          # 366-element array, 1-indexed
thresh_2024 = thresh_vals[doy_2024 - 1]     # align to target-year DOYs

# Detect heatwave days
exc_mask    = t2m_vals > thresh_2024
hw_mask     = detect_heatwaves(exc_mask, MIN_DURATION)
filled_vals = np.where(hw_mask, t2m_vals, np.nan)

# Plot
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(doy_2024, thresh_2024, color='black',     lw=2,   label=f'{PCT}th percentile threshold')
ax.plot(doy_2024, t2m_vals,   color='steelblue',  lw=1.5, label=f'Daily Tmax {TARGET_YEAR}')
ax.fill_between(
    doy_2024, thresh_2024, filled_vals,
    where=hw_mask, color='crimson', alpha=0.6, label='Heatwave',
)
ax.set_title(f'Daily Maximum Temperature and Heatwave Events — {TARGET_YEAR}',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Day of Year')
ax.set_ylabel('Temperature (C)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'heatwaves_{TARGET_YEAR}.png', dpi=150, bbox_inches='tight')
plt.show()

# Event summary
hw_dates         = t2m_2024.valid_time.values[hw_mask]
labels_2024, n24 = label(hw_mask)
print(f'Heatwave days in {TARGET_YEAR} : {hw_mask.sum()}')
print(f'Distinct events              : {n24}\n')
for ev in range(1, n24 + 1):
    idx   = np.where(labels_2024 == ev)[0]
    start = str(t2m_2024.valid_time.values[idx[0]])[:10]
    end   = str(t2m_2024.valid_time.values[idx[-1]])[:10]
    print(f'  Event {ev:2d}:  {start} to {end}  ({len(idx)} days)')

In [ ]:
# Heatwave days per calendar month
hw_dates_pd  = pd.to_datetime(hw_dates)
hw_per_month = (
    pd.Series(1, index=hw_dates_pd)
      .groupby(hw_dates_pd.month)
      .sum()
      .rename(index=lambda m: pd.Timestamp(year=int(TARGET_YEAR), month=m, day=1).strftime('%B'))
)
print(f'Heatwave days per month in {TARGET_YEAR}:\n')
print(hw_per_month.to_string())

### 5.3 · Full-Period Event Catalogue (1991–2024)

In [ ]:
records = []

for year in np.unique(data_ts['valid_time.year'].values):
    ts_yr  = data_ts.t2m.sel(valid_time=str(year))
    doy_yr = ts_yr.valid_time.dt.dayofyear.values
    t_vals = ts_yr.values
    t_thr  = thresh_vals[doy_yr - 1]

    hw          = detect_heatwaves(t_vals > t_thr, MIN_DURATION)
    grp, n      = label(hw)
    times       = ts_yr.valid_time.values
    anom        = t_vals - t_thr

    for ev in range(1, n + 1):
        idx = np.where(grp == ev)[0]
        if len(idx) < MIN_DURATION:
            continue
        intens = anom[idx]
        records.append({
            'year':             year,
            'start_date':       str(times[idx[0]])[:10],
            'end_date':         str(times[idx[-1]])[:10],
            'start_doy':        pd.to_datetime(times[idx[0]]).dayofyear,
            'end_doy':          pd.to_datetime(times[idx[-1]]).dayofyear,
            'duration':         len(idx),
            'min_intensity':    float(np.nanmin(intens)),
            'mean_intensity':   float(np.nanmean(intens)),
            'median_intensity': float(np.nanmedian(intens)),
            'max_intensity':    float(np.nanmax(intens)),
        })

df_heatwaves = pd.DataFrame(records)
print(f'Events detected: {len(df_heatwaves)}  across {df_heatwaves["year"].nunique()} years')
df_heatwaves.head(10)

---
## 6 · Results & Visualisation

### 6.1 · Heatwave Event Timeline (1991–2024)

In [ ]:
# Month tick positions (1st and 15th of each month)
tick_dates  = sorted(
    pd.date_range('2001-01-01', '2001-12-31', freq='MS').tolist() +
    pd.date_range('2001-01-15', '2001-12-15', freq='MS').tolist()
)
tick_pos    = [d.dayofyear for d in tick_dates]
tick_labels = [d.strftime('%d %b') for d in tick_dates]

fig, ax = plt.subplots(figsize=(13, 8))
for _, row in df_heatwaves.iterrows():
    ax.hlines(
        y=row['year'],
        xmin=row['start_doy'], xmax=row['end_doy'],
        color='crimson', linewidth=4, alpha=0.75,
    )
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Year', fontsize=12)
ax.set_title('Heatwave Event Timeline — Ouagadougou (1991–2024)',
             fontsize=14, fontweight='bold')
ax.set_xlim(1, 366)
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=8)
ax.set_yticks(df_heatwaves['year'].unique())
ax.invert_yaxis()
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('heatwave_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.2 · Annual Statistics

In [ ]:
# Aggregate per year, fill years with no events with zeros
grouped  = df_heatwaves.groupby('year')
df_stats = pd.DataFrame({
    'num_events':    grouped.size(),
    'max_duration':  grouped['duration'].max(),
    'med_duration':  grouped['duration'].median(),
    'avg_intensity': grouped['mean_intensity'].mean(),
    'max_intensity': grouped['max_intensity'].max(),
}).reset_index()

all_years = np.unique(data_ts['valid_time.year'].values)
df_stats  = (
    df_stats.set_index('year')
            .reindex(all_years, fill_value=0)
            .reset_index()
)
df_stats.columns.name = None
df_stats.to_csv('heatwave_annual_statistics.csv', index=False)
print(df_stats.to_string(index=False))
print('\nSaved to heatwave_annual_statistics.csv')

In [ ]:
# Horizontal bar chart: frequency, duration, intensity
x     = df_stats['year'].values
nums  = df_stats['num_events'].values
max_d = df_stats['max_duration'].values
med_d = df_stats['med_duration'].values
avg_t = df_stats['avg_intensity'].values
max_t = df_stats['max_intensity'].values

fig, axes = plt.subplots(1, 3, figsize=(16, 9), sharey=True)

axes[0].barh(x, nums, height=0.6, color='#333333', label='Number of events')
axes[0].set_xlabel('Number of heatwaves')
axes[0].set_title('Frequency', fontweight='bold')

axes[1].barh(x + 0.2, max_d, height=0.35, color='orangered', label='Max duration')
axes[1].barh(x - 0.2, med_d, height=0.35, color='#ffaa44',   label='Median duration')
axes[1].set_xlabel('Days')
axes[1].set_title('Duration', fontweight='bold')

axes[2].barh(x + 0.2, max_t, height=0.35, color='#800080', label='Max intensity')
axes[2].barh(x - 0.2, avg_t, height=0.35, color='crimson',  label='Mean intensity')
axes[2].set_xlabel('C above threshold')
axes[2].set_title('Intensity', fontweight='bold')

for ax in axes:
    ax.invert_yaxis()
    ax.set_yticks(x)
    ax.set_ylabel('Year')
    ax.legend(fontsize=9)
    ax.grid(True, axis='x', alpha=0.3)

plt.suptitle('Annual Heatwave Statistics — Ouagadougou (1991–2024)',
             fontsize=15, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('heatwave_annual_bars.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.3 · Trend Analysis

Ordinary least-squares regression of four key heatwave indicators against time. An asterisk (*) marks trends significant at the p < 0.05 level.

In [ ]:
INDICATORS = {
    'num_events':    {'label': 'Number of heatwaves',  'unit': 'events/yr', 'color': '#333333'},
    'max_duration':  {'label': 'Max duration',          'unit': 'days/yr',   'color': 'orangered'},
    'avg_intensity': {'label': 'Mean intensity',        'unit': 'C/yr',      'color': 'crimson'},
    'max_intensity': {'label': 'Max intensity',         'unit': 'C/yr',      'color': '#800080'},
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
axes = axes.flatten()

x = df_stats['year'].values
for i, (var, props) in enumerate(INDICATORS.items()):
    y = df_stats[var].values
    slope, intercept, _, p, _ = linregress(x, y)
    trend = intercept + slope * x
    sig   = ' *' if p < 0.05 else ''

    axes[i].plot(x, y, 'o-', color=props['color'], lw=1.5, ms=5, label=props['label'])
    axes[i].plot(
        x, trend, '--', color=props['color'], lw=2,
        label=f"Trend: {slope:+.2f} {props['unit']}  (p = {p:.3f}){sig}",
    )
    axes[i].set_title(props['label'], fontsize=11, fontweight='bold')
    axes[i].set_ylabel(props['label'])
    axes[i].legend(fontsize=9)
    axes[i].grid(True, alpha=0.3)

axes[2].set_xlabel('Year')
axes[3].set_xlabel('Year')

plt.suptitle(
    'Trends in Heatwave Indicators — Ouagadougou (1991–2024)\n* p < 0.05',
    fontsize=14, fontweight='bold',
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('heatwave_trends.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top-5 years for each indicator
print('Top 5 years for each heatwave indicator:\n')
for var, props in INDICATORS.items():
    top5 = df_stats.sort_values(var, ascending=False).head(5)[['year', var]]
    print(f"  {props['label']}")
    for _, row in top5.iterrows():
        print(f"    {int(row['year'])}: {row[var]:.2f}")
    print()

---
## Summary

| Output file | Description |
|---|---|
| `monthly_climatology_tmax.png` | 12-panel climatology map (1991-2020) |
| `threshold_comparison.png` | Comparison of percentile-threshold methods |
| `heatwaves_2024.png` | Daily Tmax with heatwave shading for 2024 |
| `heatwave_timeline.png` | Year-by-year event timeline (1991-2024) |
| `heatwave_annual_bars.png` | Annual statistics bar chart |
| `heatwave_trends.png` | OLS trend lines for four key indicators |
| **`heatwave_annual_statistics.csv`** | **Analysis-ready table of annual statistics** |

---

### Methodology Note

The **90th-percentile threshold** is computed from the 1991-2020 baseline following the WMO standard climatological normal. The groupby-quantile estimate is smoothed with a 30-day centred rolling mean to produce a continuous seasonal cycle, avoiding step-function artefacts. Events require a minimum of **3 consecutive days** of exceedance (Perkins & Alexander, 2013). Intensity is expressed as degrees Celsius above the local threshold, accounting for the seasonal cycle.

### References

- Munoz Sabater, J. (2021). ERA5-Land hourly data from 1950 to present. *Copernicus Climate Change Service (C3S) Climate Data Store (CDS)*. https://doi.org/10.24381/cds.e2161bac
- Perkins, S. E., & Alexander, L. V. (2013). On the measurement of heat waves. *Journal of Climate*, 26(13), 4500-4517. https://doi.org/10.1175/JCLI-D-12-00383.1
- WMO (2017). *WMO Guidelines on the Calculation of Climate Normals*. WMO-No. 1203.